# Phase 4 — Deep Learning & Computer Vision (Transfer Learning)

## Background

`03_ml_modeling.ipynb` (Phase 3) fit classical machine learning models on twelve
hand-engineered pixel, texture, edge, and region features — summary statistics computed from
each chest X-ray, not the raw image itself. This notebook is Phase 4 of the capstone
methodology: instead of hand-crafting features, a convolutional neural network learns its own
hierarchy of spatial filters directly from the raw pixels via transfer learning. It fine-tunes
`lab/src/model.py`'s ResNet18 — pretrained on ImageNet, with its final layer replaced by a
single logit output (`P(pneumonia)`) — on the same `train`/`val` split used throughout this
project, reusing the training loop already implemented in `lab/src/train.py` rather than
reimplementing it here.

`train.py` trains the model and tracks validation AUC each epoch, but computes no other metric
and never touches `test`. This notebook adds exactly that: after training, it loads the saved
checkpoint, runs it once over the held-out `test` split, and computes the identical metric
suite Phase 3 used (Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix), saving the raw
predictions so `05_model_comparison.ipynb` can compare Phase 3's hand-engineered-feature models
against this end-to-end deep-learning model on equal footing.

### Imports and configuration

This notebook reuses `lab/src/dataset.py` (`PneumoniaXrayDataset`, `build_transform`, the
224×224 resize, ImageNet normalization, and the `NORMAL=0`/`PNEUMONIA=1` class index order),
`lab/src/model.py` (`build_model`), and `lab/src/ood.py` (the out-of-distribution guardrail
built later in this notebook) instead of redefining any of them — this is the exact
preprocessing/architecture pairing `train.py` uses to produce the checkpoint below, and that
eventually gets exported to the serving app, so it must never drift into a second,
notebook-local copy. `SEED = 42` matches `03_ml_modeling.ipynb`'s constant, fixed here for this
notebook's own `torch`/`numpy` RNGs at test time; `train.py` itself has no `--seed` flag, so
this does not make the training run below deterministic — a limitation inherited from the
existing script, not introduced here.

In [1]:
import sys
sys.path.insert(0, "../src")

from dataset import PneumoniaXrayDataset, build_transform
from model import build_model
from train import train
from ood import K_NEIGHBORS, config_dict, is_grayscale_like, knn_distance, reference_stats, standardize

from argparse import Namespace
import json
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

### Train the ResNet18 model

Rather than reimplementing the training loop, this cell calls `train()` from the existing
`lab/src/train.py` directly (already imported above), which trains for the given number of
epochs (10, matching the source decision), tracking validation AUC each epoch on `../data/val`
(the 18-image split — a known, accepted limitation for early-stopping reliability, revisited
in the Conclusion below). It saves the checkpoint with the best `val_auc` to
`../artifacts/model_best.pt`, then exports it to `../artifacts/model.onnx` (with both the
usual classification logit and the model's 512-dim penultimate-layer embedding as outputs)
and saves every `train` image's embedding to `../artifacts/train_embeddings.npy` — the
reference set the out-of-distribution guardrail built later in this notebook compares
against. Reusing `train()` here keeps exactly one training implementation for Phase 4,
instead of two that could drift apart.

This is a long-running model-fitting operation and is intentionally left un-executed in this
notebook — the project owner runs it separately, once ready. `train.py` has no `--seed` flag,
so re-running this cell will not reproduce byte-identical results.

In [2]:
# Trains ResNet18 for 10 epochs, tracking validation AUC on ../data/val each epoch, keeps
# the best checkpoint, and exports it to ONNX. This is a model-fitting run — intentionally
# NOT executed as part of this notebook.
train_args = Namespace(
    train_dir="../data/train",
    val_dir="../data/val",
    epochs=10,
    batch_size=32,
    lr=1e-4,
    output="../artifacts/model_best.pt",
    onnx_output="../artifacts/model.onnx",
    embeddings_output="../artifacts/train_embeddings.npy",
)
train(train_args)

Training on: mps
epoch 1/10 train_loss=0.0944 val_loss=0.1377 val_auc=1.0000
epoch 2/10 train_loss=0.0246 val_loss=0.5805 val_auc=1.0000
epoch 3/10 train_loss=0.0130 val_loss=0.0098 val_auc=1.0000
epoch 4/10 train_loss=0.0170 val_loss=0.0463 val_auc=1.0000
epoch 5/10 train_loss=0.0123 val_loss=0.1379 val_auc=1.0000
epoch 6/10 train_loss=0.0026 val_loss=0.1183 val_auc=1.0000
epoch 7/10 train_loss=0.0030 val_loss=0.0254 val_auc=1.0000
epoch 8/10 train_loss=0.0017 val_loss=0.0269 val_auc=1.0000
epoch 9/10 train_loss=0.0012 val_loss=0.1080 val_auc=1.0000
epoch 10/10 train_loss=0.0122 val_loss=0.3169 val_auc=1.0000


/Users/niltonconstantino/personal/workspace.personal/data-science/pneumonia-project/ml-service/lab/notebooks/../src/train.py:38: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ResNetWithEmbedding([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNetWithEmbedding([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/opt/homebrew/anaconda3/envs/pneumonia-lab/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX: ../artifacts/model.onnx (outputs: logits, 512-dim embedding)
Train embeddings: ../artifacts/train_embeddings.npy (5216 x 512)

Best val_auc=1.0000. Checkpoint: ../artifacts/model_best.pt.
To serve this model, see 'Promoting a model' in lab/README.md.


### Test-set evaluation

`train.py` only tracks AUC on `val` during training — it never touches `test` and computes no
other metric. This section is new logic, not duplicated from `train.py`: it rebuilds the same
ResNet18 architecture without downloading ImageNet weights again (the fine-tuned weights are
about to be loaded from the checkpoint instead), restores the trained weights, and runs a
forward pass over every image in `../data/test` (624 images, never used for training or model
selection) to collect predicted probabilities. Those probabilities are then reduced to the same
metric suite Phase 3 used — Accuracy, Precision, Recall, and F1 at the standard 0.5 probability
threshold, plus the threshold-independent ROC-AUC and the confusion matrix (rows = true class,
columns = predicted class, `NORMAL=0`/`PNEUMONIA=1` order) — so the two modeling approaches can
be compared metric-for-metric in `05_model_comparison.ipynb`.

In [3]:
model = build_model(pretrained=False)
model.load_state_dict(torch.load("../artifacts/model_best.pt", map_location="cpu"))
model.eval()

test_dataset = PneumoniaXrayDataset("../data/test")
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

y_true, y_pred_proba = [], []
with torch.no_grad():
    for images, targets in test_loader:
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        y_pred_proba.extend(probs.tolist())
        y_true.extend(targets.cpu().numpy().ravel().tolist())

y_true = np.array(y_true, dtype=int)
y_pred_proba = np.array(y_pred_proba, dtype=float)
image_ids = [img_path.name for img_path, _ in test_dataset.samples]

len(image_ids), len(y_true), len(y_pred_proba)

(624, 624, 624)

In [4]:
def compute_metrics(y_true, y_pred_proba, threshold=0.5):
    y_pred = (y_pred_proba >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_pred_proba),
        "Confusion Matrix": confusion_matrix(y_true, y_pred),
    }


metrics_resnet18 = compute_metrics(y_true, y_pred_proba)

for key, value in metrics_resnet18.items():
    if key == "Confusion Matrix":
        print("Confusion Matrix (rows=true, cols=pred, order=[NORMAL, PNEUMONIA]):")
        print(value)
    else:
        print(f"{key}: {value:.4f}")

Accuracy: 0.8301
Precision: 0.7874
Recall: 0.9974
F1: 0.8801
ROC-AUC: 0.9422
Confusion Matrix (rows=true, cols=pred, order=[NORMAL, PNEUMONIA]):
[[129 105]
 [  1 389]]


### Save test-set predictions

Predicted probabilities on `test`, together with the ground-truth labels, are persisted to
`../artifacts/predictions/phase4_test_predictions.csv` so `05_model_comparison.ipynb` can
compare this ResNet18 against Phase 3's M4/M6 without refitting or rerunning anything — all
three models' predictions are recomputed from saved CSVs alone, on the exact same `test`
images. The column names (`Image_ID`, `y_true`, `y_pred_proba_resnet18`) mirror
`03_ml_modeling.ipynb`'s `phase3_test_predictions.csv` schema exactly.

In [5]:
predictions_dir = Path("../artifacts/predictions")
predictions_dir.mkdir(parents=True, exist_ok=True)

predictions_df = pd.DataFrame(
    {
        "Image_ID": image_ids,
        "y_true": y_true,
        "y_pred_proba_resnet18": y_pred_proba,
    }
)
predictions_df.to_csv(predictions_dir / "phase4_test_predictions.csv", index=False)

print(
    f"Saved {len(predictions_df)} predictions to "
    f"{predictions_dir / 'phase4_test_predictions.csv'}"
)

Saved 624 predictions to ../artifacts/predictions/phase4_test_predictions.csv


### Out-of-distribution guardrail: why and how

The metrics above are all computed on real chest X-rays. The served model
(`app/model/inference.py`) doesn't get that guarantee — it has to handle whatever gets
uploaded, and a sigmoid always outputs *something*, no matter how unlike a chest X-ray the
input is. The cell below shows this is not a hypothetical: unrelated images turn into a
confident "pneumonia" prediction with no guardrail in place. The rest of this section builds
and validates the two independent checks `lab/src/ood.py` implements to catch that before an
image reaches this model:

1. **Grayscale check** — real chest X-rays in this dataset are stored with `R == G == B`
   exactly per pixel (grayscale scans replicated across 3 JPEG channels); a genuine color
   photo will not be.
2. **k-NN embedding distance** — catches grayscale-but-not-an-X-ray content (random noise, a
   structureless pattern) the first check can't see, by measuring distance — in this model's
   own 512-dim penultimate-layer embedding space — to every `train` image's embedding. This
   is the nearest-neighbor method from M4 Unit 5 (K-Nearest Neighbors): Euclidean distance on
   standardized features, same as taught there, applied here to anomaly thresholding instead
   of majority-vote classification.

In [6]:
onnx_session = ort.InferenceSession("../artifacts/model.onnx", providers=["CPUExecutionProvider"])
transform = build_transform()


def onnx_predict(pil_image):
    tensor = transform(pil_image).unsqueeze(0).numpy()
    logits, embedding = onnx_session.run(["logits", "embedding"], {"input": tensor})
    probability = 1 / (1 + np.exp(-logits.ravel()[0]))
    return float(probability), embedding[0]


demo_images = {
    "random noise (color)": Image.fromarray(rng.integers(0, 256, (224, 224, 3), dtype=np.uint8)),
    "solid red": Image.new("RGB", (224, 224), (255, 0, 0)),
    "solid white": Image.new("RGB", (224, 224), (255, 255, 255)),
}

for name, image in demo_images.items():
    probability, _ = onnx_predict(image)
    print(f"{name:25s} P(pneumonia)={probability:.4f}")

random noise (color)      P(pneumonia)=0.9962
solid red                 P(pneumonia)=0.9348
solid white               P(pneumonia)=0.9790


#### Guardrail 1 — grayscale check

`is_grayscale_like` measures, per pixel, the standard deviation across the R/G/B channels. A
chest X-ray scores ~0 everywhere (see `lab/src/test_ood.py`, verified against all 5856 images
in `lab/data/{train,val,test}/`); a color image does not.

In [7]:
for name, image in demo_images.items():
    print(f"{name:25s} is_grayscale_like={is_grayscale_like(image)}")

real_xray_path, _ = test_dataset.samples[0]
print(f"{'real X-ray (for contrast)':25s} is_grayscale_like={is_grayscale_like(Image.open(real_xray_path))}")

random noise (color)      is_grayscale_like=False
solid red                 is_grayscale_like=False
solid white               is_grayscale_like=True
real X-ray (for contrast) is_grayscale_like=True


#### Guardrail 2 — k-NN distance in embedding space

The grayscale check has a blind spot: a grayscale image that isn't an X-ray at all (random
noise, a repeating pattern) still passes it. `knn_distance` closes that gap by comparing, in
the model's own embedding space, how far a new image sits from its `k` nearest neighbors among
every `train` image's embedding — the same nearest-neighbor method M4 Unit 5 teaches (Euclidean
distance, standardized features), reused here to threshold anomalies instead of voting on a
class.

`K_NEIGHBORS = 10` (`lab/src/ood.py`) is deliberately smaller than the course's classification
rule of thumb (`k ≈ sqrt(n)`, ≈72 for this project's 5216 training images): that rule balances
bias and variance for majority-vote classification, not sensitivity to local structure for
anomaly thresholding, where a smaller `k` stays closer to genuinely nearby points instead of
averaging in more distant ones.

To set a rejection threshold, the cell below computes, for a sample of `train` images, the
leave-one-out `k`-NN distance to the *rest* of `train` — i.e. how far a genuine training image
typically sits from its own nearest neighbors. The 99th percentile of that distribution is used
as the threshold: comfortably outside where real X-rays cluster, without being so tight that
ordinary variation between real X-rays gets rejected. It also writes `../artifacts/ood_config.json`
— `promote.py` reads this and folds it into `app/model/manifest.yaml`'s `ood` section, which
`app/model/inference.py` reads at startup. This threshold is data-dependent (it comes from
*this* run's own embeddings), so it belongs in that published contract, not hardcoded twice in
`lab/src/ood.py` and `app/model/inference.py`.

In [8]:
train_embeddings = np.load("../artifacts/train_embeddings.npy")
ref_mean, ref_std = reference_stats(train_embeddings)
train_standardized = standardize(train_embeddings, ref_mean, ref_std)

sample_idx = rng.choice(len(train_embeddings), size=1000, replace=False)
loo_distances = []
for i in sample_idx:
    distances = np.linalg.norm(train_standardized - train_standardized[i][None, :], axis=1)
    distances[i] = np.inf  # exclude the point itself
    distances.sort()
    loo_distances.append(distances[:K_NEIGHBORS].mean())
loo_distances = np.array(loo_distances)

for p in [50, 90, 95, 99, 100]:
    print(f"p{p}: {np.percentile(loo_distances, p):.2f}")

KNN_DISTANCE_THRESHOLD = round(float(np.percentile(loo_distances, 99)), 1)
print(f"\nChosen threshold (p99 of leave-one-out train distances): {KNN_DISTANCE_THRESHOLD}")

ood_config = config_dict(KNN_DISTANCE_THRESHOLD)
ood_config_path = Path("../artifacts/ood_config.json")
ood_config_path.write_text(json.dumps(ood_config, indent=2))
print(f"Saved {ood_config_path}: {ood_config}")

p50: 13.12
p90: 15.23
p95: 16.05
p99: 17.70
p100: 21.51

Chosen threshold (p99 of leave-one-out train distances): 17.7
Saved ../artifacts/ood_config.json: {'max_channel_std': 2.0, 'min_grayscale_fraction': 0.95, 'k_neighbors': 10, 'knn_distance_threshold': 17.7}


#### Validating the threshold

Three checks against `KNN_DISTANCE_THRESHOLD`: the color images from the confidence demo above
(already caught by the grayscale check — shown here only to confirm the k-NN distance agrees),
two grayscale-but-not-an-X-ray patterns constructed specifically to test the gap the grayscale
check leaves open, and a sample of real held-out `test` X-rays, which must **not** be rejected
— a screening tool that blocks real patients is worse than one that lets an occasional strange
image through.

In [9]:
def knn_verdict(image):
    _, embedding = onnx_predict(image)
    distance = knn_distance(embedding, train_standardized, ref_mean, ref_std)
    verdict = "REJECTED" if distance > KNN_DISTANCE_THRESHOLD else "accepted"
    return distance, verdict


print("-- color images (grayscale check already rejects these) --")
for name, image in demo_images.items():
    distance, verdict = knn_verdict(image)
    print(f"{name:25s} knn_distance={distance:6.2f} -> {verdict}")

print("\n-- grayscale-but-not-an-X-ray (the gap the grayscale check leaves open) --")
grayscale_noise = Image.fromarray(
    np.repeat(rng.integers(0, 256, (224, 224), dtype=np.uint8)[:, :, None], 3, axis=2)
)
checkerboard_pattern = np.indices((224, 224)).sum(axis=0) % 32 < 16
grayscale_checkerboard = Image.fromarray(
    np.repeat((checkerboard_pattern * 255).astype(np.uint8)[:, :, None], 3, axis=2)
)
for name, image in {
    "grayscale random noise": grayscale_noise,
    "grayscale checkerboard": grayscale_checkerboard,
}.items():
    distance, verdict = knn_verdict(image)
    print(f"{name:25s} knn_distance={distance:6.2f} -> {verdict}")

print("\n-- real held-out test X-rays (must all be 'accepted') --")
false_rejections = 0
sample_paths = [test_dataset.samples[i][0] for i in rng.choice(len(test_dataset), size=20, replace=False)]
for path in sample_paths:
    distance, verdict = knn_verdict(Image.open(path).convert("RGB"))
    if verdict == "REJECTED":
        false_rejections += 1
    print(f"{path.name:35s} knn_distance={distance:6.2f} -> {verdict}")

print(f"\n{false_rejections}/{len(sample_paths)} real test X-rays incorrectly rejected")

-- color images (grayscale check already rejects these) --
random noise (color)      knn_distance= 69.27 -> REJECTED
solid red                 knn_distance=126.49 -> REJECTED
solid white               knn_distance= 17.36 -> accepted

-- grayscale-but-not-an-X-ray (the gap the grayscale check leaves open) --
grayscale random noise    knn_distance= 35.85 -> REJECTED
grayscale checkerboard    knn_distance= 35.72 -> REJECTED

-- real held-out test X-rays (must all be 'accepted') --
person101_bacteria_483.jpeg         knn_distance= 13.40 -> accepted
person45_virus_95.jpeg              knn_distance= 12.85 -> accepted
NORMAL2-IM-0353-0001.jpeg           knn_distance= 14.54 -> accepted
NORMAL2-IM-0285-0001.jpeg           knn_distance= 15.95 -> accepted
person134_bacteria_642.jpeg         knn_distance= 13.08 -> accepted
NORMAL2-IM-0271-0001.jpeg           knn_distance= 15.57 -> accepted
person128_bacteria_605.jpeg         knn_distance= 11.34 -> accepted
person136_bacteria_652.jpeg         knn_d

### Conclusion

**Classification performance.** On the held-out 624-image `test` split, this ResNet18
achieves Accuracy 0.8301, Precision 0.7874, Recall 0.9974, F1 0.8801, ROC-AUC 0.9422
(confusion matrix above: 129 true negatives, 105 false positives, 1 false negative, 389 true
positives). In a pneumonia screening context, **Recall** is the metric that matters most —
missing a true positive is more costly than a false alarm — and this model misses only 1 of
390 PNEUMONIA cases. The trade-off shows up in Precision: 105 of the 234 truly `NORMAL`
images get flagged `PNEUMONIA`, which a screening pipeline is meant to absorb via a
follow-up read, not something with the same cost as a missed case.

**On the val-based early stopping (18 images):** the training log shows `val_auc` reaching
its ceiling of 1.0000 at epoch 1 and staying there through epoch 10 — the split is small
enough to be perfectly separable, so past epoch 1 it stopped providing any signal to
distinguish between checkpoints. Combined with `train()`'s strict `>` comparison, the
checkpoint promoted to `model_best.pt` is actually epoch 1's, not one chosen by genuine
competition across all 10 epochs. That epoch 1's weights still generalize this well to
`test` (ROC-AUC 0.9422) says more about how fast ImageNet-pretrained features adapt to this
task (`train_loss` was already down to 0.0944 after epoch 1) than about the early-stopping
logic, which never actually got exercised past the first epoch.

Worth noting: this run reproduced the *exact* same test metrics as the training run this
notebook originally shipped with, down to the confusion matrix. `train.py` has no `--seed`
flag of its own, but this notebook's `torch.manual_seed(SEED)` (set above for the test-time
`torch` RNG) also happens to seed `train()`'s `DataLoader(shuffle=True)` and the replaced
`fc` layer's initialization, since `train()` runs in this same kernel and inherits `torch`'s
already-seeded global RNG state — an incidental, not designed-for, source of
reproducibility.

**Out-of-distribution guardrail.** The confidence demo confirms the problem is real: random
noise, solid red, and solid white all score 93–99% "pneumonia" confidence with no guardrail.
The grayscale check correctly rejects the two color images; the k-NN check (threshold 17.7,
the 99th percentile of 1000 training images' own leave-one-out distance — see the percentile
table above) correctly rejects the grayscale random-noise and checkerboard patterns the
grayscale check can't see, and correctly accepts 19 of 20 sampled real `test` X-rays.

Two honest gaps this run surfaced, not swept under the rug:

- **Solid white passes both guardrails** (`is_grayscale_like=True`, since R==G==B trivially
  holds for any achromatic color, not just real grayscale scans; `knn_distance=17.36`, just
  under the 17.7 threshold) — a flat white or black image would still reach the model today.
  Neither check was designed to catch a degenerate, textureless image specifically; a third,
  cheap guardrail (e.g. rejecting near-zero pixel-to-pixel variance, which no real X-ray has)
  would close this gap and is a natural next step, not yet implemented.
- **1 of the 20 sampled real `test` X-rays was rejected** (`person94_bacteria_457.jpeg`,
  distance 18.00, just 0.3 over the threshold) — and it is a true PNEUMONIA case the model
  itself would have flagged with 99.95% confidence had the guardrail let it through. This is
  the exact cost the threshold's design already anticipated ("a screening tool that blocks
  real patients is worse than one that lets an occasional strange image through"), now
  measured concretely: roughly a 1-in-20 rate on this sample, in the same ballpark as the
  ~1% rate a p99 threshold implies. Loosening the threshold (e.g. to p99.9, or the observed
  max of 21.51) would trade this away against admitting more OOD content — a call for the
  project owner, not resolved here.

`phase4_test_predictions.csv` and `ood_config.json`, both saved above, are what
`05_model_comparison.ipynb` and `promote.py` respectively depend on next (see
`lab/README.md`'s "Promoting a model" for the publishing mechanics) — `05` needs a re-run now
that its input file has been refreshed.